In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [5]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [6]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='7bceb615-e05d-4faa-8b5e-3e8c7da8e8a1'),
              AIMessage(content='<think>\nOkay, the user is asking for the square root of 456. Let me check which tool I can use here. The available functions are call_subagent_1 for square root and call_subagent_2 for square. Since they want the square root, I should use call_subagent_1. The parameter required is x, which is 456. So I need to structure the tool call with name call_subagent_1 and arguments { "x": 456 }. Let me make sure there are no typos. Yep, that should do it.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-25T14:41:40.2683329Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5529195200, 'load_duration': 2062980500, 'prompt_eval_count': 239, 'prompt_eval_duration': 340168100, 'eval_count': 148, 'eval_duration': 3123428800, 'logprobs': None, 'model_nam